In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("lidiapierre/fr_sexism_labelled")

# Check the dataset structure
print(dataset)
print(dataset["train"].features)

# Look at a few examples to understand the data format
for i in range(3):
    print(f"Example {i}:")
    print(f"Sentence: {dataset['train'][i]['Sentences']}")
    print(f"Label: {dataset['train'][i]['Label']}")
    print("---")

# Check the label distribution
import pandas as pd
import numpy as np
labels = np.array(dataset["train"]["Label"])
print(f"Label distribution: {np.unique(labels, return_counts=True)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/955 [00:00<?, ?B/s]

(…)-00000-of-00001-2e804b729bcc0e0b.parquet:   0%|          | 0.00/120k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1137 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Sentences', 'Label', 'fr_sentences'],
        num_rows: 1137
    })
})
{'Unnamed: 0': Value(dtype='int64', id=None), 'Sentences': Value(dtype='string', id=None), 'Label': Value(dtype='int64', id=None), 'fr_sentences': Value(dtype='string', id=None)}
Example 0:
Sentence: For a woman, that is good
Label: 1
---
Example 1:
Sentence: Just dress sexy and you will sign any contract.
Label: 1
---
Example 2:
Sentence: Are you having period ?
Label: 1
---
Label distribution: (array([0, 1]), array([513, 624]))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Use BERT base uncased for English text
model_name = "bert-base-uncased"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Define a function to tokenize the inputs
def tokenize_function(examples):
    return tokenizer(
        examples["Sentences"],  # Use the English sentences
        padding="max_length",
        truncation=True,
        max_length=128  # Shorter sequences to save memory
    )

# Process in smaller batches to save memory
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    batch_size=64,  # Smaller batch size
    num_proc=2      # Limit processes
)

# Add proper labels column
def prepare_labels(examples):
    # Convert the "Label" column to the "labels" format expected by the model
    examples["labels"] = examples["Label"]
    return examples

tokenized_datasets = tokenized_datasets.map(
    prepare_labels,
    batched=True,
    batch_size=64
)

# Keep only necessary columns
columns_to_keep = ["input_ids", "token_type_ids", "attention_mask", "labels"]
tokenized_datasets = tokenized_datasets.remove_columns(
    [col for col in tokenized_datasets["train"].column_names if col not in columns_to_keep]
)

# Convert to PyTorch format
tokenized_datasets.set_format("torch")

# Create a train-test split if there's no test set
if "test" not in tokenized_datasets:
    split_datasets = tokenized_datasets["train"].train_test_split(test_size=0.2, seed=42)
    tokenized_datasets = {"train": split_datasets["train"], "test": split_datasets["test"]}

Map (num_proc=2):   0%|          | 0/1137 [00:00<?, ? examples/s]

Map:   0%|          | 0/1137 [00:00<?, ? examples/s]

In [ ]:
# Check the number of unique labels
unique_labels = set(dataset["train"]["Label"])
num_labels = len(unique_labels)
print(f"Number of unique labels: {num_labels}")
print(f"Label values: {unique_labels}")

# Load the model with from_tf=True
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    from_tf=True
)

Number of unique labels: 2
Label values: {0, 1}


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

All TF 2.0 model weights were used when initializing BertForSequenceClassification.

All the weights of BertForSequenceClassification were initialized from the TF 2.0 model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BertForSequenceClassification for predictions without further training.


In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
import gc

# Free up memory
gc.collect()
torch.cuda.empty_cache()

# Define the metric calculation function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Determine if binary or multi-class
    average = 'binary' if num_labels == 2 else 'macro'

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=average)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Define training arguments optimized for memory
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,    # Smaller batch size
    per_device_eval_batch_size=8,     # Smaller batch size
    num_train_epochs=4,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    # Memory optimization settings
    fp16=True,                        # Mixed precision
    gradient_accumulation_steps=4,    # Accumulate gradients
    gradient_checkpointing=True,      # Trade compute for memory
    report_to="none",                 # Disable logging to save memory
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

# Save the model
model.save_pretrained("./sexism_detection_model")
tokenizer.save_pretrained("./sexism_detection_model")

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
0,No log,0.432009,0.842105,0.859375,0.846154,0.873016
1,No log,0.375975,0.864035,0.875502,0.886179,0.865079
2,No log,0.357653,0.864035,0.874494,0.892562,0.857143


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
0,No log,0.432009,0.842105,0.859375,0.846154,0.873016
1,No log,0.375975,0.864035,0.875502,0.886179,0.865079
2,No log,0.357653,0.864035,0.874494,0.892562,0.857143
3,No log,0.351268,0.872807,0.883534,0.894309,0.873016


Evaluation results: {'eval_loss': 0.35126757621765137, 'eval_accuracy': 0.8728070175438597, 'eval_f1': 0.8835341365461847, 'eval_precision': 0.8943089430894309, 'eval_recall': 0.873015873015873, 'eval_runtime': 86.5261, 'eval_samples_per_second': 2.635, 'eval_steps_per_second': 0.335, 'epoch': 3.982456140350877}


('./sexism_detection_model/tokenizer_config.json',
 './sexism_detection_model/special_tokens_map.json',
 './sexism_detection_model/vocab.txt',
 './sexism_detection_model/added_tokens.json',
 './sexism_detection_model/tokenizer.json')

In [ ]:
def predict_sexism(text_list):
    # Tokenize inputs
    inputs = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=128
    )

    # Move inputs to the same device as the model
    device = model.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Make prediction
    with torch.no_grad():
        outputs = model(**inputs)

    # Get prediction labels
    predictions = torch.argmax(outputs.logits, dim=-1)

    # Convert to human-readable labels
    # Note: Adjust these based on your actual label meanings
    label_map = {0: "Not Sexist", 1: "Sexist"}  # Update this based on your dataset

    results = []
    for pred in predictions:
        results.append(label_map[pred.item()])

    return results

# Test with examples
test_texts = [
    "Women don't belong in leadership positions.",
    "Everyone deserves equal opportunities regardless of gender.",
]
predictions = predict_sexism(test_texts)
for text, pred in zip(test_texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred}")
    print("---")

Text: Women don't belong in leadership positions.
Prediction: Sexist
---
Text: Everyone deserves equal opportunities regardless of gender.
Prediction: Not Sexist
---


In [ ]:
test_texts = [
    "women are terrible",
    "girls should be educated",
    "girls should not be educated",
    "everyone is dumb",
    "everyone deserves education"
]
predictions = predict_sexism(test_texts)
for text, pred in zip(test_texts, predictions):
    print(f"Text: {text}")
    print(f"Prediction: {pred}")
    print("---")

Text: women are terrible
Prediction: Sexist
---
Text: girls should be educated
Prediction: Not Sexist
---
Text: girls should not be educated
Prediction: Sexist
---
Text: everyone is dumb
Prediction: Not Sexist
---
Text: everyone deserves education
Prediction: Not Sexist
---


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
model.save_pretrained("/content/drive/MyDrive/AI_Project/SexismDetection/model/")
tokenizer.save_pretrained("/content/drive/MyDrive/AI_Project/SexismDetection/model/")

('/content/drive/MyDrive/AI_Project/SexismDetection/model/tokenizer_config.json',
 '/content/drive/MyDrive/AI_Project/SexismDetection/model/special_tokens_map.json',
 '/content/drive/MyDrive/AI_Project/SexismDetection/model/vocab.txt',
 '/content/drive/MyDrive/AI_Project/SexismDetection/model/added_tokens.json',
 '/content/drive/MyDrive/AI_Project/SexismDetection/model/tokenizer.json')

In [ ]:
!zip -r sexism_detection_model.zip ./sexism_detection_model

  adding: sexism_detection_model/ (stored 0%)
  adding: sexism_detection_model/tokenizer_config.json (deflated 75%)
  adding: sexism_detection_model/tokenizer.json (deflated 71%)
  adding: sexism_detection_model/special_tokens_map.json (deflated 42%)
  adding: sexism_detection_model/model.safetensors (deflated 7%)
  adding: sexism_detection_model/vocab.txt (deflated 53%)
  adding: sexism_detection_model/config.json (deflated 49%)


In [ ]:
from google.colab import files
files.download("sexism_detection_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/AI_Project/SexismDetection/model/"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)


In [ ]:
from transformers import pipeline

# Create pipeline
clf_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer)

# Use it
clf_pipeline("She is a terrible leader.")


Device set to use cpu


[{'label': 'LABEL_0', 'score': 0.9360009431838989}]

In [ ]:
clf_pipeline("She is overreacting")

[{'label': 'LABEL_1', 'score': 0.8983380794525146}]

In [ ]:
test = ["She can't be a doctor, she should be a nurse", "Anyone can be a doctor"]
for text in test:
    print(clf_pipeline(text))

[{'label': 'LABEL_1', 'score': 0.9184212684631348}]
[{'label': 'LABEL_0', 'score': 0.7461972832679749}]
